In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce_silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Hardcoded list of tables for testing
table_list = ["users", "buyers", "sellers", "countries"]

# Define reusable transformation functions
def transform_users(df):
    return (df.withColumn("countrycode", upper(col("countrycode")))
              .withColumn("language_full",
                          expr("CASE WHEN language = 'EN' THEN 'English' "
                               "WHEN language = 'FR' THEN 'French' ELSE 'Other' END"))
              .withColumn("gender", when(col("gender").startswith("M"), "Male")
                                    .when(col("gender").startswith("F"), "Female")
                                    .otherwise("Other"))
              .withColumn("account_age_years", round(col("seniority")/365,2)))

def transform_buyers(df):
    return (df.withColumn("country", initcap(col("country")))
              .withColumn("female_to_male_ratio",
                          round(col("femalebuyers")/(col("malebuyers")+1),2))
              .withColumn("wishlist_to_purchase_ratio",
                          round(col("totalproductswished")/(col("totalproductsbought")+1),2)))

def transform_sellers(df):
    mean_pass_rate = df.select(round(avg("meansellerpassrate"),2).alias("avg")).collect()[0]["avg"]
    return (df.withColumn("country", initcap(col("country")))
              .withColumn("sex", upper(col("sex")))
              .withColumn("seller_size_category",
                          when(col("nbsellers")<500,"Small")
                          .when(col("nbsellers")<2000,"Medium")
                          .otherwise("Large"))
              .withColumn("meansellerpassrate",
                          when(col("meansellerpassrate").isNull(), mean_pass_rate)
                          .otherwise(col("meansellerpassrate"))))

def transform_countries(df):
    return (df.withColumn("country", initcap(col("country")))
              .withColumn("performance_indicator",
                          round(col("toptotalproductssold")/(col("toptotalproductslisted")+1),2))
              .withColumn("activity_level",
                          when(col("meanofflinedays")<30,"Highly Active")
                          .when(col("meanofflinedays")<60,"Moderately Active")
                          .otherwise("Low Activity")))

# Loop through each table in the list
for table_name in table_list:
    bronze_table = f"ecommerce.{table_name}_raw"
    silver_table = f"ecommerce.{table_name}_silver"

    df = spark.table(bronze_table)

    # Dispatch transformations
    if table_name == "users":
        df = transform_users(df)
    elif table_name == "buyers":
        df = transform_buyers(df)
    elif table_name == "sellers":
        df = transform_sellers(df)
    elif table_name == "countries":
        df = transform_countries(df)

    # Write to Silver
    df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(silver_table)

    print(f"Processed {table_name} → {silver_table}")
